# Step 1 = Create the environment

In [1]:
from statsbombpy import sb 

comps = sb.competitions()
comps.head()

/Users/hugoninho/shotmap-messi/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882


### In the following cell, I want to display only the **liga matches** with the 3 columns : season_name, season_id & competition_id

In [2]:
liga = comps[comps.competition_name == 'La Liga']
liga[['competition_id', 'season_name', 'season_id']]


,competition_id,season_name,season_id
40,11,2020/2021,90
41,11,2019/2020,42
42,11,2018/2019,4
43,11,2017/2018,1
44,11,2016/2017,2
45,11,2015/2016,27
46,11,2014/2015,26
47,11,2013/2014,25
48,11,2012/2013,24
49,11,2011/2012,23


### I want to focus on the 2011/2012 Messi's season 

In [3]:
matches = sb.matches(competition_id=11, season_id=23)
print(len(matches))
matches.sort_values("match_date")[["match_date", "home_team", "away_team", "home_score", "away_score"]].head(7) # sort value to get chronological order of matches

/Users/hugoninho/shotmap-messi/.venv/lib/python3.13/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


37


,match_date,home_team,away_team,home_score,away_score
28,2011-08-29,Barcelona,Villarreal,5,0
18,2011-09-10,Real Sociedad,Barcelona,2,2
0,2011-09-17,Barcelona,Osasuna,8,0
16,2011-09-21,Valencia,Barcelona,2,2
11,2011-09-24,Barcelona,Atlético Madrid,5,0
21,2011-10-02,Sporting Gijón,Barcelona,0,1
15,2011-10-15,Barcelona,Racing Santander,3,0


### I want to control that Messi is really present during theses matches 

In [12]:
match_ids = matches["match_id"].tolist()
lineups = sb.lineups(match_id=match_ids[16])
lineups["Barcelona"][["player_name", "jersey_number"]]

,player_name,jersey_number
0,Francesc Fàbregas i Soler,4
1,Pedro Eliezer Rodríguez Ledesma,17
2,Daniel Alves da Silva,2
3,Sergio Busquets i Burgos,16
4,Thiago Alcântara do Nascimento,11
5,Lionel Andrés Messi Cuccittini,10
6,Javier Alejandro Mascherano,14
7,David Villa Sánchez,7
8,Carles Puyol i Saforcada,5
9,Xavier Hernández Creus,6


# Step 2 = Extract all Messi's shots 

In [5]:
import warnings 
warnings.filterwarnings("ignore") # to ignore warnings from statsbombpy

In [14]:
match_ids = matches["match_id"].tolist()
event = sb.events(match_id=match_ids[16])
print(event.columns)

Index(['bad_behaviour_card', 'ball_receipt_outcome',
       'ball_recovery_recovery_failure', 'block_deflection', 'block_offensive',
       'carry_end_location', 'clearance_aerial_won', 'clearance_body_part',
       'clearance_head', 'clearance_left_foot', 'clearance_right_foot',
       'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun',
       'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage',
       'foul_committed_card', 'foul_committed_type', 'foul_won_advantage',
       'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location',
       'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_technique',
       'goalkeeper_type', 'id', 'index', 'injury_stoppage_in_chain',
       'interception_outcome', 'location', 'match_id', 'minute', 'off_camera',
       'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id',
       'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected',
       'pass_end_location', 'pass_goal_

In [15]:
event['type'].value_counts()

type
Pass                 1066
Ball Receipt*         991
Carry                 925
Pressure              356
Ball Recovery         100
Duel                   73
Clearance              48
Dribble                46
Dribbled Past          34
Block                  33
Dispossessed           32
Interception           30
Foul Committed         25
Foul Won               24
Goal Keeper            21
Miscontrol             19
Shot                   18
Substitution            6
Half Start              4
Injury Stoppage         4
Half End                4
Tactical Shift          3
Starting XI             2
Own Goal Against        1
Own Goal For            1
Offside                 1
Player Off              1
Player On               1
Referee Ball-Drop       1
Bad Behaviour           1
Name: count, dtype: int64

### These are the different events types 

In [8]:
shots = event [
(event['type'] == 'Shot') & 
(event['player'] .str.contains('Messi', na=False))
]
shots[['minute', 'location', 'shot_statsbomb_xg', 'shot_outcome', 'shot_body_part']]


,minute,location,shot_statsbomb_xg,shot_outcome,shot_body_part
4487,8,"[99.6, 40.8]",0.064231,Blocked,Left Foot
4496,32,"[107.0, 40.8]",0.128680,Goal,Head
4500,50,"[103.8, 42.7]",0.140421,Goal,Left Foot
4501,73,"[94.0, 36.2]",0.069230,Post,Left Foot
4503,80,"[113.2, 30.6]",0.279085,Goal,Left Foot
4508,89,"[108.6, 41.9]",0.170313,Saved,Head
4509,90,"[111.6, 49.9]",0.194485,Saved,Left Foot


### this table doesn't show me in which game these events occured, so i have to had the match paper in it

In [9]:
infos_match = matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]]

shots = shots.merge(infos_match, on="match_id", how="left")

# here we add the matches information to the shots dataframe, so we can have the match date, home team, away team, home score and away score for each shot taken by Messi.

In [10]:
import numpy as np

shots["adversaire"] = np.where(shots["home_team"] == "Barcelona",
                               shots["away_team"],
                               shots["home_team"])

shots["domicile"] = shots["home_team"] == "Barcelona"

In [11]:
shots[["match_date", "adversaire", "domicile", "minute",
       "shot_statsbomb_xg", "shot_outcome", "shot_body_part"]]

,match_date,adversaire,domicile,minute,shot_statsbomb_xg,shot_outcome,shot_body_part
0,2012-01-22,Málaga,False,8,0.064231,Blocked,Left Foot
1,2012-01-22,Málaga,False,32,0.128680,Goal,Head
2,2012-01-22,Málaga,False,50,0.140421,Goal,Left Foot
3,2012-01-22,Málaga,False,73,0.069230,Post,Left Foot
4,2012-01-22,Málaga,False,80,0.279085,Goal,Left Foot
5,2012-01-22,Málaga,False,89,0.170313,Saved,Head
6,2012-01-22,Málaga,False,90,0.194485,Saved,Left Foot


### So now we have every shots events of Messi in a specific match (here it's Malaga), let's observe these events on the different matches of the season